In [1]:
!pip install transformers sentencepiece accelerate openpyxl -q

In [16]:
import pandas as pd
import re
import ast
from copy import deepcopy
from transformers import pipeline

INPUT_FILE = "train_br_ready_strict.xlsx"
OUTPUT_FILE = "train_br_augmented.xlsx"
OUTPUT_AUG_ONLY = "train_br_only_augmented.xlsx"

TEXT_COL = "text_ar"
GLOSS_COL = "gloss_ar"

df = pd.read_excel(INPUT_FILE)

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
df.head(10)

Shape: (102, 15)
Columns: ['id', 'text_ar', 'gloss_ar', 'source_type', 'aug_method', 'parent_id', 'is_augmented', 'text_norm', 'gloss_norm', 'tokens_text', 'tokens_gloss', 'replaceable_positions', 'replaceable_tokens', 'protected_tokens_found', 'br_ready']


,id,text_ar,gloss_ar,source_type,aug_method,parent_id,is_augmented,text_norm,gloss_norm,tokens_text,tokens_gloss,replaceable_positions,replaceable_tokens,protected_tokens_found,br_ready
0,1,انتبه,انتبه خطر تمام,original,none,1,0,انتبه,انتبه خطر تمام,['انتبه'],"['انتبه', 'خطر', 'تمام']",[],[],['انتبه'],0
1,2,حبة 3 مرة قبل الاكل بنص ساعة .,3 مرة اكل قبل نص ساعة حبة,original,none,2,0,حبة 3 مرة قبل الاكل بنص ساعة .,3 مرة اكل قبل نص ساعة حبة,"['حبة', '3', 'مرة', 'قبل', 'الاكل', 'بنص', 'سا...","['3', 'مرة', 'اكل', 'قبل', 'نص', 'ساعة', 'حبة']",[4],['الاكل'],"['بنص', 'حبة', 'ساعة', 'قبل', 'مرة']",1
2,3,حبة 3 مرة قبل الاكل .,3 مرة اكل قبل حبة,original,none,3,0,حبة 3 مرة قبل الاكل .,3 مرة اكل قبل حبة,"['حبة', '3', 'مرة', 'قبل', 'الاكل', '.']","['3', 'مرة', 'اكل', 'قبل', 'حبة']",[4],['الاكل'],"['حبة', 'قبل', 'مرة']",1
3,4,حبة 3 مرة قبل الاكل بساعة .,3 مرة اكل قبل ساعة حبة,original,none,4,0,حبة 3 مرة قبل الاكل بساعة .,3 مرة اكل قبل ساعة حبة,"['حبة', '3', 'مرة', 'قبل', 'الاكل', 'بساعة', '.']","['3', 'مرة', 'اكل', 'قبل', 'ساعة', 'حبة']",[4],['الاكل'],"['بساعة', 'حبة', 'قبل', 'مرة']",1
4,6,6 شهر,6 شهر,original,none,6,0,6 شهر,6 شهر,"['6', 'شهر']","['6', 'شهر']",[],[],['شهر'],0
5,7,نقطتين بكل طرف,اذن يمين نقطة - نقطة اذن يسار نقطة - نقطة,original,none,7,0,نقطتين بكل طرف,اذن يمين نقطة - نقطة اذن يسار نقطة - نقطة,"['نقطتين', 'بكل', 'طرف']","['اذن', 'يمين', 'نقطة', '-', 'نقطة', 'اذن', 'ي...",[],[],"['بكل', 'نقطتين']",0
6,8,لمدة ثلاثة يوم,استمرار ثلاثة يوم بس تمام,original,none,8,0,لمدة ثلاثة يوم,استمرار ثلاثة يوم بس تمام,"['لمدة', 'ثلاثة', 'يوم']","['استمرار', 'ثلاثة', 'يوم', 'بس', 'تمام']",[],[],"['ثلاثة', 'لمدة', 'يوم']",0
7,9,خده ورا الاكل,اكل بعد حبة تمام معدة فاضية لا,original,none,9,0,خده ورا الاكل,اكل بعد حبة تمام معدة فاضية لا,"['خده', 'ورا', 'الاكل']","['اكل', 'بعد', 'حبة', 'تمام', 'معدة', 'فاضية',...",[2],['الاكل'],['ورا'],1
8,10,بعد الاكل,اكل خلص فورا دواء تمام,original,none,10,0,بعد الاكل,اكل خلص فورا دواء تمام,"['بعد', 'الاكل']","['اكل', 'خلص', 'فورا', 'دواء', 'تمام']",[1],['الاكل'],['بعد'],1
9,13,"ممكن يعملك شوية اسهال , لا تقلق .",انتبه تمام جسم تعب تمام اسهال امساك دوخة تمام,original,none,13,0,"ممكن يعملك شوية اسهال , لا تقلق .",انتبه تمام جسم تعب تمام اسهال امساك دوخة تمام,"['ممكن', 'يعملك', 'شوية', 'اسهال', ',', 'لا', ...","['انتبه', 'تمام', 'جسم', 'تعب', 'تمام', 'اسهال...",[],[],"['اسهال', 'تقلق', 'شوية', 'لا', 'ممكن', 'يعملك']",0


In [17]:
def normalize_ar(text: str) -> str:
    text = str(text).strip()

    text = re.sub(r"[ًٌٍَُِّْـ]", "", text)
    text = re.sub(r"[إأآا]", "ا", text)
    text = re.sub(r"ى", "ي", text)
    text = re.sub(r"ؤ", "و", text)
    text = re.sub(r"ئ", "ي", text)

    # فصل الواو عن الكلمات القابلة للاستبدال
    text = re.sub(
        r"\bو(الاكل|اكل|الطعام|طعام|وجبة|وجبات|الفطور|فطور|الغدا|غدا|العشا|عشا|خذ|تناول|ابلع|حبة|قرص|كبسولة|كبسوله)\b",
        r"و \1",
        text
    )

    text = re.sub(r"\s+", " ", text).strip()
    return text

def tokenize(text: str):
    return normalize_ar(text).split()

def detokenize(tokens):
    text = " ".join(tokens)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def parse_list_cell(x):
    if isinstance(x, list):
        return x
    if pd.isna(x):
        return []
    if isinstance(x, str):
        x = x.strip()
        if not x:
            return []
        try:
            val = ast.literal_eval(x)
            return val if isinstance(val, list) else []
        except Exception:
            return []
    return []

In [18]:
SAFE_REPLACEMENTS = {
    # أكل / وجبات
    "الاكل": ["الطعام", "وجبة"],
    "اكل": ["الطعام", "وجبة"],
    "الطعام": ["الاكل", "وجبة"],
    "طعام": ["الاكل", "وجبة"],
    "وجبة": ["الاكل", "الطعام"],
    "وجبات": ["الاكل", "الطعام"],

    "الفطور": ["الاكل"],
    "فطور": ["الاكل"],
    "الغدا": ["الاكل"],
    "غدا": ["الاكل"],
    "العشا": ["الاكل"],
    "عشا": ["الاكل"],

    # أفعال التناول
    "خذ": ["تناول", "ابلع"],
    "خد": ["تناول", "ابلع"],
    "تناول": ["خذ", "ابلع"],
    "ابلع": ["خذ", "تناول"],

    # شكل الجرعة
    "حبة": ["قرص", "كبسولة"],
    "حبه": ["قرص", "كبسولة"],
    "قرص": ["حبة", "كبسولة"],
    "كبسولة": ["حبة", "قرص"],
    "كبسوله": ["حبة", "قرص"],
}

SAFE_REPLACEMENTS = {
    normalize_ar(k): [normalize_ar(v) for v in vals]
    for k, vals in SAFE_REPLACEMENTS.items()
}

SAFE_REPLACEMENTS

{'الاكل': ['الطعام', 'وجبة'],
 'اكل': ['الطعام', 'وجبة'],
 'الطعام': ['الاكل', 'وجبة'],
 'طعام': ['الاكل', 'وجبة'],
 'وجبة': ['الاكل', 'الطعام'],
 'وجبات': ['الاكل', 'الطعام'],
 'الفطور': ['الاكل'],
 'فطور': ['الاكل'],
 'الغدا': ['الاكل'],
 'غدا': ['الاكل'],
 'العشا': ['الاكل'],
 'عشا': ['الاكل'],
 'خذ': ['تناول', 'ابلع'],
 'خد': ['تناول', 'ابلع'],
 'تناول': ['خذ', 'ابلع'],
 'ابلع': ['خذ', 'تناول'],
 'حبة': ['قرص', 'كبسولة'],
 'حبه': ['قرص', 'كبسولة'],
 'قرص': ['حبة', 'كبسولة'],
 'كبسولة': ['حبة', 'قرص'],
 'كبسوله': ['حبة', 'قرص']}

In [19]:
TEXT_TO_GLOSS_MAP = {
    # أكل / وجبات
    "الاكل": "اكل",
    "اكل": "اكل",
    "الطعام": "اكل",
    "طعام": "اكل",
    "وجبة": "اكل",
    "وجبات": "اكل",
    "الفطور": "فطور",
    "فطور": "فطور",
    "الغدا": "غدا",
    "غدا": "غدا",
    "العشا": "عشا",
    "عشا": "عشا",

    # أفعال
    "خذ": "اخذ",
    "خد": "اخذ",
    "تناول": "تناول",
    "ابلع": "بلع",

    # أشكال دوائية
    "حبة": "حبة",
    "حبه": "حبة",
    "قرص": "قرص",
    "كبسولة": "كبسولة",
    "كبسوله": "كبسولة",
}

TEXT_TO_GLOSS_MAP = {
    normalize_ar(k): normalize_ar(v)
    for k, v in TEXT_TO_GLOSS_MAP.items()
}

TEXT_TO_GLOSS_MAP

{'الاكل': 'اكل',
 'اكل': 'اكل',
 'الطعام': 'اكل',
 'طعام': 'اكل',
 'وجبة': 'اكل',
 'وجبات': 'اكل',
 'الفطور': 'فطور',
 'فطور': 'فطور',
 'الغدا': 'غدا',
 'غدا': 'غدا',
 'العشا': 'عشا',
 'عشا': 'عشا',
 'خذ': 'اخذ',
 'خد': 'اخذ',
 'تناول': 'تناول',
 'ابلع': 'بلع',
 'حبة': 'حبة',
 'حبه': 'حبة',
 'قرص': 'قرص',
 'كبسولة': 'كبسولة',
 'كبسوله': 'كبسولة'}

In [20]:
FILL_MASK_MODEL = "aubmindlab/araelectra-base-generator"

fill_mask = pipeline(
    "fill-mask",
    model=FILL_MASK_MODEL,
    tokenizer=FILL_MASK_MODEL
)

print("Fill-mask model loaded.")
print("Mask token:", fill_mask.tokenizer.mask_token)

Loading weights:   0%|          | 0/204 [00:00<?, ?it/s]

ElectraForMaskedLM LOAD REPORT from: aubmindlab/araelectra-base-generator
Key                             | Status     |  | 
--------------------------------+------------+--+-
electra.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Fill-mask model loaded.
Mask token: [MASK]


In [21]:
def clean_candidate_token(tok: str) -> str:
    tok = str(tok).strip()
    tok = tok.replace("##", "")
    tok = normalize_ar(tok)
    tok = re.sub(r"[^\w\u0600-\u06FF]", "", tok).strip()
    return tok

def get_allowed_candidates(original_token: str):
    original_token = normalize_ar(original_token)
    return SAFE_REPLACEMENTS.get(original_token, [])

def build_masked_text(tokens, position, mask_token):
    temp = tokens.copy()
    temp[position] = mask_token
    return detokenize(temp)

def propose_candidates_hybrid(masked_text: str, original_token: str, top_k=20):
    original_token = normalize_ar(original_token)
    allowed = get_allowed_candidates(original_token)

    # إذا ما في بدائل يدوية، نوقف
    if not allowed:
        return []

    # ابدأ دائمًا بالبدائل اليدوية
    manual_candidates = [{"candidate_token": c, "score": 1.0, "source": "manual"} for c in allowed]

    # ثم حاول fill-mask
    try:
        preds = fill_mask(masked_text, top_k=top_k)
    except Exception:
        preds = []

    model_candidates = []
    seen = set()

    for p in preds:
        tok = clean_candidate_token(p.get("token_str", ""))
        if tok in allowed and tok not in seen and tok != original_token:
            seen.add(tok)
            model_candidates.append({
                "candidate_token": tok,
                "score": float(p.get("score", 0.0)),
                "source": "fill_mask"
            })

    # دمج اليدوي مع ما وافق عليه المودل
    merged = []
    used = set()

    # أولًا ما وافق عليه المودل
    for item in sorted(model_candidates, key=lambda x: x["score"], reverse=True):
        key = item["candidate_token"]
        if key not in used:
            used.add(key)
            merged.append(item)

    # ثم أكمل بالبدائل اليدوية الناقصة
    for item in manual_candidates:
        key = item["candidate_token"]
        if key not in used and key != original_token:
            used.add(key)
            merged.append(item)

    return merged

In [22]:
def replace_first_match(tokens, old_value, new_value):
    old_value = normalize_ar(old_value)
    new_value = normalize_ar(new_value)

    updated = []
    replaced = False

    for tok in tokens:
        tok_norm = normalize_ar(tok)
        if not replaced and tok_norm == old_value:
            updated.append(new_value)
            replaced = True
        else:
            updated.append(tok_norm)

    return updated, replaced


def update_gloss(gloss_text: str, old_text_token: str, new_text_token: str) -> str:
    gloss_tokens = tokenize(gloss_text)

    old_gloss = TEXT_TO_GLOSS_MAP.get(normalize_ar(old_text_token))
    new_gloss = TEXT_TO_GLOSS_MAP.get(normalize_ar(new_text_token))

    # إذا ما عرفنا التحويل، نرجع الغلوس كما هو
    if not old_gloss or not new_gloss:
        return gloss_text

    updated_tokens, replaced = replace_first_match(gloss_tokens, old_gloss, new_gloss)

    if replaced:
        return detokenize(updated_tokens)
    else:
        return gloss_text

In [23]:
df["replaceable_positions"] = df["replaceable_positions"].apply(parse_list_cell)
df["replaceable_tokens"] = df["replaceable_tokens"].apply(parse_list_cell)

df_ready = df[df["br_ready"] == 1].copy()

print("Total rows:", len(df))
print("Rows ready for BR:", len(df_ready))

df_ready[["id", TEXT_COL, "replaceable_positions", "replaceable_tokens"]].head(20)

Total rows: 102
Rows ready for BR: 14


,id,text_ar,replaceable_positions,replaceable_tokens
1,2,حبة 3 مرة قبل الاكل بنص ساعة .,[4],[الاكل]
2,3,حبة 3 مرة قبل الاكل .,[4],[الاكل]
3,4,حبة 3 مرة قبل الاكل بساعة .,[4],[الاكل]
7,9,خده ورا الاكل,[2],[الاكل]
8,10,بعد الاكل,[1],[الاكل]
44,58,"قبل الفطور عالريق , خود حبة كل يوم .",[1],[الفطور]
48,62,"بعد وجبة دسمة عالصبحية , متل البيض والحليب وزي...",[1],[وجبة]
49,63,حبة قبل الفطور والغدا والعشا بنص ساعة .,"[2, 4, 6]","[الفطور, الغدا, العشا]"
55,71,خود حبة يوميا بعد الاكل,[4],[الاكل]
56,72,حبة الصبح بعد الفطور,[3],[الفطور]


In [24]:
augmented_rows = []
new_sample_id_start = 100000
current_new_id = new_sample_id_start

for _, row in df_ready.iterrows():
    row_id = int(row["id"])
    text = str(row[TEXT_COL]).strip()
    gloss = str(row[GLOSS_COL]).strip()

    tokens = tokenize(text)
    positions = row["replaceable_positions"]

    for pos in positions:
        if pos >= len(tokens):
            continue

        original_token = normalize_ar(tokens[pos])

        # لا نكمل إذا لا يوجد بدائل آمنة
        allowed = get_allowed_candidates(original_token)
        if not allowed:
            continue

        masked_text = build_masked_text(tokens, pos, fill_mask.tokenizer.mask_token)
        candidates = propose_candidates_hybrid(masked_text, original_token, top_k=20)

        for cand in candidates:
            new_token = normalize_ar(cand["candidate_token"])

            if new_token == original_token:
                continue

            new_tokens = tokens.copy()
            new_tokens[pos] = new_token
            new_text = detokenize(new_tokens)

            new_gloss = update_gloss(gloss, original_token, new_token)

            new_row = deepcopy(row.to_dict())
            new_row["sample_id"] = current_new_id
            new_row["parent_id"] = row_id
            new_row["source_type"] = "augmented"
            new_row["aug_method"] = "BR"
            new_row["is_augmented"] = 1

            new_row[TEXT_COL] = new_text
            new_row[GLOSS_COL] = new_gloss

            new_row["masked_text"] = masked_text
            new_row["replace_position"] = int(pos)
            new_row["replaced_token_original"] = original_token
            new_row["replaced_token_new"] = new_token
            new_row["replacement_score"] = float(cand["score"])
            new_row["replacement_source"] = cand["source"]

            # تحديث الأعمدة المشتقة
            new_row["text_norm"] = normalize_ar(new_text)
            new_row["gloss_norm"] = normalize_ar(new_gloss)
            new_row["tokens_text"] = str(tokenize(new_text))
            new_row["tokens_gloss"] = str(tokenize(new_gloss))

            # هذه الأعمدة تخص مرحلة التحضير، لا العينة النهائية
            new_row["replaceable_positions"] = "[]"
            new_row["replaceable_tokens"] = "[]"
            new_row["br_ready"] = 0

            augmented_rows.append(new_row)
            current_new_id += 1

print("Generated augmented rows:", len(augmented_rows))

Generated augmented rows: 28


In [25]:
aug_df = pd.DataFrame(augmented_rows)

print("Augmented shape:", aug_df.shape)
aug_df.head(20)

Augmented shape: (28, 22)


,id,text_ar,gloss_ar,source_type,aug_method,parent_id,is_augmented,text_norm,gloss_norm,tokens_text,...,replaceable_tokens,protected_tokens_found,br_ready,sample_id,masked_text,replace_position,replaced_token_original,replaced_token_new,replacement_score,replacement_source
0,2,حبة 3 مرة قبل الطعام بنص ساعة .,3 مرة اكل قبل نص ساعة حبة,augmented,BR,2,1,حبة 3 مرة قبل الطعام بنص ساعة .,3 مرة اكل قبل نص ساعة حبة,"['حبة', '3', 'مرة', 'قبل', 'الطعام', 'بنص', 'س...",...,[],"['بنص', 'حبة', 'ساعة', 'قبل', 'مرة']",0,100000,حبة 3 مرة قبل [MASK] بنص ساعة .,4,الاكل,الطعام,1.000000,manual
1,2,حبة 3 مرة قبل وجبة بنص ساعة .,3 مرة اكل قبل نص ساعة حبة,augmented,BR,2,1,حبة 3 مرة قبل وجبة بنص ساعة .,3 مرة اكل قبل نص ساعة حبة,"['حبة', '3', 'مرة', 'قبل', 'وجبة', 'بنص', 'ساع...",...,[],"['بنص', 'حبة', 'ساعة', 'قبل', 'مرة']",0,100001,حبة 3 مرة قبل [MASK] بنص ساعة .,4,الاكل,وجبة,1.000000,manual
2,3,حبة 3 مرة قبل الطعام .,3 مرة اكل قبل حبة,augmented,BR,3,1,حبة 3 مرة قبل الطعام .,3 مرة اكل قبل حبة,"['حبة', '3', 'مرة', 'قبل', 'الطعام', '.']",...,[],"['حبة', 'قبل', 'مرة']",0,100002,حبة 3 مرة قبل [MASK] .,4,الاكل,الطعام,1.000000,manual
3,3,حبة 3 مرة قبل وجبة .,3 مرة اكل قبل حبة,augmented,BR,3,1,حبة 3 مرة قبل وجبة .,3 مرة اكل قبل حبة,"['حبة', '3', 'مرة', 'قبل', 'وجبة', '.']",...,[],"['حبة', 'قبل', 'مرة']",0,100003,حبة 3 مرة قبل [MASK] .,4,الاكل,وجبة,1.000000,manual
4,4,حبة 3 مرة قبل الطعام بساعة .,3 مرة اكل قبل ساعة حبة,augmented,BR,4,1,حبة 3 مرة قبل الطعام بساعة .,3 مرة اكل قبل ساعة حبة,"['حبة', '3', 'مرة', 'قبل', 'الطعام', 'بساعة', ...",...,[],"['بساعة', 'حبة', 'قبل', 'مرة']",0,100004,حبة 3 مرة قبل [MASK] بساعة .,4,الاكل,الطعام,0.011844,fill_mask
5,4,حبة 3 مرة قبل وجبة بساعة .,3 مرة اكل قبل ساعة حبة,augmented,BR,4,1,حبة 3 مرة قبل وجبة بساعة .,3 مرة اكل قبل ساعة حبة,"['حبة', '3', 'مرة', 'قبل', 'وجبة', 'بساعة', '.']",...,[],"['بساعة', 'حبة', 'قبل', 'مرة']",0,100005,حبة 3 مرة قبل [MASK] بساعة .,4,الاكل,وجبة,1.000000,manual
6,9,خده ورا الطعام,اكل بعد حبة تمام معدة فاضية لا,augmented,BR,9,1,خده ورا الطعام,اكل بعد حبة تمام معدة فاضية لا,"['خده', 'ورا', 'الطعام']",...,[],['ورا'],0,100006,خده ورا [MASK],2,الاكل,الطعام,1.000000,manual
7,9,خده ورا وجبة,اكل بعد حبة تمام معدة فاضية لا,augmented,BR,9,1,خده ورا وجبة,اكل بعد حبة تمام معدة فاضية لا,"['خده', 'ورا', 'وجبة']",...,[],['ورا'],0,100007,خده ورا [MASK],2,الاكل,وجبة,1.000000,manual
8,10,بعد الطعام,اكل خلص فورا دواء تمام,augmented,BR,10,1,بعد الطعام,اكل خلص فورا دواء تمام,"['بعد', 'الطعام']",...,[],['بعد'],0,100008,بعد [MASK],1,الاكل,الطعام,1.000000,manual
9,10,بعد وجبة,اكل خلص فورا دواء تمام,augmented,BR,10,1,بعد وجبة,اكل خلص فورا دواء تمام,"['بعد', 'وجبة']",...,[],['بعد'],0,100009,بعد [MASK],1,الاكل,وجبة,1.000000,manual


In [26]:
if len(aug_df) > 0:
    before_dedup = len(aug_df)
    aug_df = aug_df.drop_duplicates(subset=[TEXT_COL, GLOSS_COL]).reset_index(drop=True)
    after_dedup = len(aug_df)

    print("Before dedup:", before_dedup)
    print("After dedup :", after_dedup)
else:
    print("No augmented rows generated.")

Before dedup: 28
After dedup : 28


In [27]:
base_train_df = df.copy()

if "sample_id" not in base_train_df.columns:
    base_train_df["sample_id"] = base_train_df["id"]

combined_train_df = pd.concat([base_train_df, aug_df], ignore_index=True)

print("Base train rows    :", len(base_train_df))
print("Augmented train rows:", len(aug_df))
print("Combined train rows :", len(combined_train_df))

Base train rows    : 102
Augmented train rows: 28
Combined train rows : 130


In [28]:
preferred_cols = [
    "sample_id",
    "id",
    "parent_id",
    "source_type",
    "aug_method",
    "is_augmented",
    TEXT_COL,
    GLOSS_COL,
    "text_norm",
    "gloss_norm",
    "tokens_text",
    "tokens_gloss",
    "masked_text",
    "replace_position",
    "replaced_token_original",
    "replaced_token_new",
    "replacement_score",
    "replacement_source",
    "replaceable_positions",
    "replaceable_tokens",
    "protected_tokens_found",
    "br_ready"
]

existing_preferred = [c for c in preferred_cols if c in combined_train_df.columns]
remaining_cols = [c for c in combined_train_df.columns if c not in existing_preferred]

combined_train_df = combined_train_df[existing_preferred + remaining_cols]
aug_df = aug_df[[c for c in existing_preferred + remaining_cols if c in aug_df.columns]]

combined_train_df.head(20)

,sample_id,id,parent_id,source_type,aug_method,is_augmented,text_ar,gloss_ar,text_norm,gloss_norm,...,masked_text,replace_position,replaced_token_original,replaced_token_new,replacement_score,replacement_source,replaceable_positions,replaceable_tokens,protected_tokens_found,br_ready
0,1,1,1,original,none,0,انتبه,انتبه خطر تمام,انتبه,انتبه خطر تمام,...,NaN,NaN,NaN,NaN,NaN,NaN,[],[],['انتبه'],0
1,2,2,2,original,none,0,حبة 3 مرة قبل الاكل بنص ساعة .,3 مرة اكل قبل نص ساعة حبة,حبة 3 مرة قبل الاكل بنص ساعة .,3 مرة اكل قبل نص ساعة حبة,...,NaN,NaN,NaN,NaN,NaN,NaN,[4],[الاكل],"['بنص', 'حبة', 'ساعة', 'قبل', 'مرة']",1
2,3,3,3,original,none,0,حبة 3 مرة قبل الاكل .,3 مرة اكل قبل حبة,حبة 3 مرة قبل الاكل .,3 مرة اكل قبل حبة,...,NaN,NaN,NaN,NaN,NaN,NaN,[4],[الاكل],"['حبة', 'قبل', 'مرة']",1
3,4,4,4,original,none,0,حبة 3 مرة قبل الاكل بساعة .,3 مرة اكل قبل ساعة حبة,حبة 3 مرة قبل الاكل بساعة .,3 مرة اكل قبل ساعة حبة,...,NaN,NaN,NaN,NaN,NaN,NaN,[4],[الاكل],"['بساعة', 'حبة', 'قبل', 'مرة']",1
4,6,6,6,original,none,0,6 شهر,6 شهر,6 شهر,6 شهر,...,NaN,NaN,NaN,NaN,NaN,NaN,[],[],['شهر'],0
5,7,7,7,original,none,0,نقطتين بكل طرف,اذن يمين نقطة - نقطة اذن يسار نقطة - نقطة,نقطتين بكل طرف,اذن يمين نقطة - نقطة اذن يسار نقطة - نقطة,...,NaN,NaN,NaN,NaN,NaN,NaN,[],[],"['بكل', 'نقطتين']",0
6,8,8,8,original,none,0,لمدة ثلاثة يوم,استمرار ثلاثة يوم بس تمام,لمدة ثلاثة يوم,استمرار ثلاثة يوم بس تمام,...,NaN,NaN,NaN,NaN,NaN,NaN,[],[],"['ثلاثة', 'لمدة', 'يوم']",0
7,9,9,9,original,none,0,خده ورا الاكل,اكل بعد حبة تمام معدة فاضية لا,خده ورا الاكل,اكل بعد حبة تمام معدة فاضية لا,...,NaN,NaN,NaN,NaN,NaN,NaN,[2],[الاكل],['ورا'],1
8,10,10,10,original,none,0,بعد الاكل,اكل خلص فورا دواء تمام,بعد الاكل,اكل خلص فورا دواء تمام,...,NaN,NaN,NaN,NaN,NaN,NaN,[1],[الاكل],['بعد'],1
9,13,13,13,original,none,0,"ممكن يعملك شوية اسهال , لا تقلق .",انتبه تمام جسم تعب تمام اسهال امساك دوخة تمام,"ممكن يعملك شوية اسهال , لا تقلق .",انتبه تمام جسم تعب تمام اسهال امساك دوخة تمام,...,NaN,NaN,NaN,NaN,NaN,NaN,[],[],"['اسهال', 'تقلق', 'شوية', 'لا', 'ممكن', 'يعملك']",0


In [29]:
aug_df.to_excel(OUTPUT_AUG_ONLY, index=False)
combined_train_df.to_excel(OUTPUT_FILE, index=False)

print("Saved:")
print("-", OUTPUT_AUG_ONLY)
print("-", OUTPUT_FILE)

Saved:
- train_br_only_augmented.xlsx
- train_br_augmented.xlsx


In [30]:
preview_cols = [
    "parent_id",
    TEXT_COL,
    GLOSS_COL,
    "masked_text",
    "replaced_token_original",
    "replaced_token_new",
    "replacement_score",
    "replacement_source"
]

aug_df[preview_cols].head(30)

,parent_id,text_ar,gloss_ar,masked_text,replaced_token_original,replaced_token_new,replacement_score,replacement_source
0,2,حبة 3 مرة قبل الطعام بنص ساعة .,3 مرة اكل قبل نص ساعة حبة,حبة 3 مرة قبل [MASK] بنص ساعة .,الاكل,الطعام,1.000000,manual
1,2,حبة 3 مرة قبل وجبة بنص ساعة .,3 مرة اكل قبل نص ساعة حبة,حبة 3 مرة قبل [MASK] بنص ساعة .,الاكل,وجبة,1.000000,manual
2,3,حبة 3 مرة قبل الطعام .,3 مرة اكل قبل حبة,حبة 3 مرة قبل [MASK] .,الاكل,الطعام,1.000000,manual
3,3,حبة 3 مرة قبل وجبة .,3 مرة اكل قبل حبة,حبة 3 مرة قبل [MASK] .,الاكل,وجبة,1.000000,manual
4,4,حبة 3 مرة قبل الطعام بساعة .,3 مرة اكل قبل ساعة حبة,حبة 3 مرة قبل [MASK] بساعة .,الاكل,الطعام,0.011844,fill_mask
5,4,حبة 3 مرة قبل وجبة بساعة .,3 مرة اكل قبل ساعة حبة,حبة 3 مرة قبل [MASK] بساعة .,الاكل,وجبة,1.000000,manual
6,9,خده ورا الطعام,اكل بعد حبة تمام معدة فاضية لا,خده ورا [MASK],الاكل,الطعام,1.000000,manual
7,9,خده ورا وجبة,اكل بعد حبة تمام معدة فاضية لا,خده ورا [MASK],الاكل,وجبة,1.000000,manual
8,10,بعد الطعام,اكل خلص فورا دواء تمام,بعد [MASK],الاكل,الطعام,1.000000,manual
9,10,بعد وجبة,اكل خلص فورا دواء تمام,بعد [MASK],الاكل,وجبة,1.000000,manual


In [31]:
if len(aug_df) > 0:
    print("Original tokens:")
    print(sorted(aug_df["replaced_token_original"].dropna().astype(str).unique().tolist()))

    print("\nNew tokens:")
    print(sorted(aug_df["replaced_token_new"].dropna().astype(str).unique().tolist()))

    print("\nSources:")
    print(aug_df["replacement_source"].value_counts())
else:
    print("No augmented rows generated.")

Original tokens:
['الاكل', 'العشا', 'الغدا', 'الفطور', 'وجبة']

New tokens:
['الاكل', 'الطعام', 'وجبة']

Sources:
replacement_source
manual       25
fill_mask     3
Name: count, dtype: int64


In [32]:
aug_df[[TEXT_COL, GLOSS_COL, "replaced_token_original", "replaced_token_new"]].head(30)

,text_ar,gloss_ar,replaced_token_original,replaced_token_new
0,حبة 3 مرة قبل الطعام بنص ساعة .,3 مرة اكل قبل نص ساعة حبة,الاكل,الطعام
1,حبة 3 مرة قبل وجبة بنص ساعة .,3 مرة اكل قبل نص ساعة حبة,الاكل,وجبة
2,حبة 3 مرة قبل الطعام .,3 مرة اكل قبل حبة,الاكل,الطعام
3,حبة 3 مرة قبل وجبة .,3 مرة اكل قبل حبة,الاكل,وجبة
4,حبة 3 مرة قبل الطعام بساعة .,3 مرة اكل قبل ساعة حبة,الاكل,الطعام
5,حبة 3 مرة قبل وجبة بساعة .,3 مرة اكل قبل ساعة حبة,الاكل,وجبة
6,خده ورا الطعام,اكل بعد حبة تمام معدة فاضية لا,الاكل,الطعام
7,خده ورا وجبة,اكل بعد حبة تمام معدة فاضية لا,الاكل,وجبة
8,بعد الطعام,اكل خلص فورا دواء تمام,الاكل,الطعام
9,بعد وجبة,اكل خلص فورا دواء تمام,الاكل,وجبة
